# Fine-tune Cellpose on Colab
Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.

In [3]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

True Tesla T4


## 1. Get the project
Option A (git): set `REPO_URL` (private repo: put the token in Colab Secrets as `GH_TOKEN`).
Option B: leave `REPO_URL` empty and upload `project.zip` (containing `train_cellpose.py` and `strips/`) with the file panel.

In [4]:
import os, subprocess, getpass
REPO_URL = 'github.com/Mateo-Schaerer-Gonzalez/recapping_Analysis.git'   # leave '' to use an uploaded project.zip instead
REPO_URL = REPO_URL.replace('https://', '').replace('http://', '')
if REPO_URL:
    name = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
    if not os.path.exists(name):
        try:
            from google.colab import userdata
            tok = userdata.get('GH_TOKEN')
        except Exception:
            tok = ''
        if not tok:
            tok = getpass.getpass('GitHub token (read access to the repo; leave empty if the repo is public): ')
        auth = f'{tok}@' if tok else ''
        subprocess.run(['git', 'clone', f'https://{auth}{REPO_URL}'], check=True)
    else:
        subprocess.run(['git', '-C', name, 'pull'], check=True)
    os.chdir(name)
else:
    !unzip -oq /content/project.zip -d /content/project
    os.chdir('/content/project')
print(os.getcwd()); print(os.listdir('.'))

/content/recapping_Analysis
['train_colab.ipynb', 'cell_delimitation_demo.ipynb', 'train_cellpose.py', '20261109145816_SecureScan.pdf', 'strips', 'training', 'cell_detection.py', 'make_strips.py', '.git', '.gitignore', 'requirements.txt']


## 2. Install

In [5]:
!pip install -q cellpose==4.2.1.1 opencv-python-headless tifffile

## 3. Train
Uses every `strips/train/*.png` that has a corrected `_seg.npy`. Saves `models/wax_cpsam`.

In [6]:
import glob
print(len(glob.glob('strips/train/*_seg.npy')), 'corrected train strips')
os.environ['N_EPOCHS'] = '100'
!python -u train_cellpose.py

2 corrected train strips
creating new log file
[GUI INFO] : WRITING LOG OUTPUT TO /root/.cellpose/run.log

cellpose version: 	4.2.1.1 
platform:       	linux 
python version: 	3.13.16 
torch version:  	2.14.1+cu130
2026-11-11 09:21:54,483 [io INFO] WRITING LOG OUTPUT TO /root/.cellpose/run.log
2026-11-11 09:21:54,483 [io INFO] 
cellpose version: 	4.2.1.1 
platform:       	linux 
python version: 	3.13.16 
torch version:  	2.14.1+cu130
train: 2 reviewed images, 394 cells
test : 0 reviewed images
2026-11-11 09:21:54,780 [core INFO] ** TORCH CUDA version installed and working. **
2026-11-11 09:21:54,780 [core INFO] >>>> using GPU (CUDA)
2026-11-11 09:21:54,781 [models INFO] Downloading: "https://huggingface.co/mouseland/cellpose-sam/resolve/main/cpsam_v2" to /root/.cellpose/models/cpsam_v2

100% 1.15G/1.15G [00:13<00:00, 92.6MB/s]   
2026-11-11 09:22:11,218 [models INFO] >>>> loading model /root/.cellpose/models/cpsam_v2
2026-11-11 09:22:11,908 [train INFO] >>> converting bfloat16 network 

## 4. Download the model

In [10]:
from google.colab import drive
drive.mount('/content/drive')
!cp models/wax_cpsam /content/drive/MyDrive/wax_cpsam

Mounted at /content/drive


In [12]:
!cp "G:/My Drive/wax_cpsam" "c:/Users/mateo/OneDrive/Escritorio/zivildienst/agroscope part 2/recapping_Analysis/models/wax_cpsam"

cp: cannot stat 'G:/My Drive/wax_cpsam': No such file or directory
